In [ ]:
# ===== Cell 1 of 4 - set up =====
# Run this first, and again after anything restarts Python. It makes the widgets, installs only what is
# missing, gets flowR ready, and says what to do next.
import os, sys; sys.path.insert(0, os.path.abspath("engine")); import verifier
verifier.setup(dbutils)


In [ ]:
# ===== Cell 2 of 4 - your chat(), and where your files go =====
# Paste your organisation's chat() below. It must return {"answer": "<the model's reply>"}, and read the
# endpoint, token and user id with verifier.live("...") inside the function, so a fresh token is used mid-run.
import requests

def chat(SystemPrompt, MainPrompt, history=[]):
    payload = {
        "app": "sparkair",
        "enable_streaming": False,
        "flow_name": "general_chat",
        "history": history,
        "optionalParameter": {
            "maxtoken": 250000,
            "contextlength": 250000,
            "Temperature": 0.01,           # low: the same question gives the same answer
            "Top_k": 1,
            "Penalty": 1.1,
            "DefaultPrompt": SystemPrompt,
        },
        "query": MainPrompt,
        "select_all": False,
    }
    headers = {
        "Authorization": f'Bearer {verifier.live("llm_token")}',
        "SP_SSO_UID": verifier.live("reviewer_id"),
        "Content-Type": "application/json",
    }
    response = requests.post(verifier.live("llm_endpoint"), json=payload, headers=headers, timeout=180)
    response.raise_for_status()
    return response.json()          # the engine reads the reply from "answer", or from an OpenAI-shaped "choices"

verifier.check_chat(chat)           # asks it one question, then makes the Inputs folders and says what goes where


In [ ]:
# ===== Cell 3 of 4 - read the inputs and run the review =====
# Reads every input file, maps how the model computes what it returns, and asks your model to judge every
# link, here in this cell. Run it again after a token expires or the
# cluster restarts: a step already finished is never repeated.
verifier.review()


In [ ]:
# ===== Cell 4 of 4 - check the run folder against its own record =====
verifier.verify()
